# Second-Label Definition Pipeline — Reproducible Notebook (Steps 1–5)

This notebook implements an **auditable, reproducible** workflow for defining *secondary labels* from developer text.

> Generated on: 2025-09-11T05:15:35.159795Z  
> Python: 3.11.8  
> Platform: Linux-4.4.0-x86_64-with-glibc2.36

## 0. Configuration & Reproducibility

In [59]:
import re, os, json, random, hashlib, platform, datetime, pandas as pd
from pathlib import Path
random.seed(42)

# ===== Paths =====
BASE = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V1.0")  # change to your Windows path if needed
SNAPSHOT_ROOT = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots"
FILE_SAMPLE_N = 50         # how many JSONL files (repos) to sample
COMMITS_PER_REPO = 200     # how many commits per repo
#Output = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V1.0")  # change to your Windows path if needed

P_SAMPLE = BASE / "1A_sample_commits_raw.csv"
P_UNI    = BASE / "1B_corpus_top_unigrams.csv"
P_BI     = BASE / "1B_corpus_top_bigrams.csv"
P_CUES   = BASE / "1B_corpus_intent_cue_counts.csv"
P_IND_TERMS = BASE / "2_secondary_label_inductive_clusters.csv"
P_IND_SUM   = BASE / "2_secondary_label_inductive_cluster_summary.csv"
P_CODEBOOK_CSV  = BASE / "3_secondary_label_codebook_template.csv"
P_CODEBOOK_CSV_F  = BASE / "3_secondary_label_codebook_autoregex.csv"
P_CODEBOOK_YAML = BASE / "3_secondary_label_codebook_template.yaml"
P_TERMS_CSV     = BASE / "3_secondary_label_clusters_terms.csv"
P_DETECTIONS    = BASE / "4_commit_secondary_label_predictions.csv"
P_EVAL          = BASE / "5_secondary_label_evaluation.csv"

BASE.mkdir(parents=True, exist_ok=True)

def fingerprint_file(p: Path):
    if not p.exists(): return ""
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

{"env": {"python": platform.python_version(), "platform": platform.platform()}}

{'env': {'python': '3.10.3', 'platform': 'Windows-10-10.0.26100-SP0'}}

## 1A. (Optional) Build raw commit sample from JSONL snapshots

In [60]:
import json, random
from pathlib import Path
import pandas as pd

def discover_jsonl(root: str):
    if not root: return []
    rootp = Path(root)
    if not rootp.exists(): return []
    return sorted(rootp.rglob("*.jsonl"))

def repo_from_filename(p: Path) -> str:
    stem = p.stem
    if "__" in stem:
        owner, repo = stem.split("__", 1)
        return f"{owner}/{repo}"
    return stem

def split_subject_body(message: str):
    if not message: return "", ""
    lines = message.splitlines()
    subject = (lines[0] if lines else "").strip()
    body = "\n".join(lines[1:]).strip() if len(lines) > 1 else ""
    return subject, body

def get_nested(d, *keys):
    cur = d
    try:
        for k in keys: cur = cur[k]
        return cur if isinstance(cur, str) else ""
    except Exception:
        return ""

def extract_fields(rec: dict):
    sha = rec.get("sha") or get_nested(rec, "commit","sha") or get_nested(rec, "commit","tree","sha") or ""
    message = rec.get("message") or get_nested(rec, "commit","message") or rec.get("commitMessage") or ""
    subject = rec.get("subject") or ""
    body    = rec.get("body") or ""
    if not subject and message:
        subject, body = split_subject_body(message)
    pr_title = rec.get("pr_title") or get_nested(rec, "pull_request","title") or ""
    return sha, subject, body, pr_title

def reservoir_sample_jsonl(jsonl_path: Path, k: int):
    sample = []
    with open(jsonl_path, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f, start=1):
            line=line.strip()
            if not line: continue
            try:
                rec = json.loads(line)
            except Exception:
                continue
            sha, sub, body, prt = extract_fields(rec)
            row = {
                "repo": repo_from_filename(jsonl_path),
                "sha": sha, "subject": sub, "body": body, "pr_title": prt
            }
            if len(sample) < k:
                sample.append(row)
            else:
                j = random.randint(1, i)
                if j <= k: sample[j-1] = row
    return sample

FILE_SAMPLE_N = 50
COMMITS_PER_REPO = 200

if SNAPSHOT_ROOT:
    files = discover_jsonl(SNAPSHOT_ROOT)
    if files:
        if FILE_SAMPLE_N is not None and len(files) > FILE_SAMPLE_N:
            files = random.sample(files, FILE_SAMPLE_N)
        rows = []
        for p in files:
            rows.extend(reservoir_sample_jsonl(Path(p), COMMITS_PER_REPO))
        pd.DataFrame(rows).to_csv(P_SAMPLE, index=False)
        print(f"Wrote sample commits → {P_SAMPLE}  ({len(rows)} rows)")
    else:
        print("No JSONL files found; skipping resample.")
else:
    print("SNAPSHOT_ROOT not set; skipping resample.")

Wrote sample commits → C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V1.0\1A_sample_commits_raw.csv  (7601 rows)


## 1B. Corpus scan — unigrams, bigrams, intent cues

In [61]:
import re
import pandas as pd
from collections import Counter
from itertools import islice
from pathlib import Path

def must_read(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    return df

df = must_read(P_SAMPLE)

def normalize_text(s: str) -> str:
    s = str(s or "")
    s = s.lower()
    s = re.sub(r"[^\w\s\./-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["raw_text"] = (df.get("subject","").fillna("").astype(str) + " " +
                  df.get("body","").fillna("").astype(str) + " " +
                  df.get("pr_title","").fillna("").astype(str))

df["text_norm"] = df["raw_text"].apply(normalize_text)

def tokenize(text: str): return re.findall(r"[a-z0-9_]+", text.lower())
def ngrams(tokens, n=2): return list(zip(*[tokens[i:] for i in range(n)]))
def top_k(counter: Counter, k: int): return list(islice(counter.most_common(k), k))

uni = Counter(); bi = Counter()
for t in df["text_norm"]:
    toks = tokenize(t)
    uni.update(toks); bi.update(ngrams(toks, 2))

TOP_N_UNI, TOP_N_BI = 5000, 5000
top_uni = pd.DataFrame(top_k(uni, TOP_N_UNI), columns=["token","count"])
top_bi  = pd.DataFrame([(" ".join(bg), c) for bg, c in top_k(bi, TOP_N_BI)], columns=["bigram","count"])

intent_cues = [
    "migrate","migration","move","switch","port","rewrite","rework","refactor",
    "workflow","workflows",".github","gha","github","actions","ci","pipeline","circleci","travis","jenkins","gitlab","azure","ci.yml","config.yml",
    "upgrade","update","bump","adopt","raise","align","pin","agp","gradle","jdk","java","sdk","api","emulator","image","plugin","dependency","kotlin",
    "fix","unbreak","resolve","repair","address","build","compile","classpath","error","failure","failing","broken","tests",
    "add","expand","increase","enable","drop","remove","decrease","reduce","disable","shrink","prune","matrix","device","devices","abi","abis","system","target","targets",
    "speed","faster","optimize","optimise","performance","perf","cache","parallel","parallelize","parallelise","shard","sharding",
    "cleanup","clean","tidy","reorganize","restructure","rename","format","lint","revert","rollback","undo",
    "timeout","timeouts","wait","wait_for_boot","retry","retries","rerun","flake","flaky","deflake","stabilize","stability","orchestrator","ato",
    "selfhosted","self","hosted","onprem","bare","metal","custom","runner","macstadium",
    "firebase","ftl","browserstack","sauce","aws","device","farm","lab"
]
intent_cues = list(dict.fromkeys(intent_cues))

cue_counts = Counter()
for t in df["text_norm"]:
    toks = set(tokenize(t))
    for cue in intent_cues:
        if cue in toks: cue_counts[cue] += 1

cue_df = pd.DataFrame(top_k(cue_counts, len(cue_counts)), columns=["cue","rows_with_cue"])

top_uni.to_csv(P_UNI, index=False)
top_bi.to_csv(P_BI, index=False)
cue_df.to_csv(P_CUES, index=False)

print(f"Saved: {P_UNI}, {P_BI}, {P_CUES}")

Saved: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V1.0\1B_corpus_top_unigrams.csv, C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V1.0\1B_corpus_top_bigrams.csv, C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V1.0\1B_corpus_intent_cue_counts.csv


## 2. Clustering to intents — **seedless** (leader clustering)

In [62]:
import re, pandas as pd
from collections import defaultdict
from pathlib import Path

uni = pd.read_csv(P_UNI); bi = pd.read_csv(P_BI); cues = pd.read_csv(P_CUES)
for d in (uni, bi, cues):
    d.columns = [c.strip().lower() for c in d.columns]

df = pd.read_csv(P_SAMPLE)
df.columns = [c.strip().lower() for c in df.columns]

def norm_text(s: str):
    s = str(s or "").lower()
    s = re.sub(r"[^\w\s\./-]+", " ", s); s = re.sub(r"\s+", " ", s).strip()
    return s
def toks(s): return re.findall(r"[a-z0-9_]+", s)
def bigr(tokens): return ["{} {}".format(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]

df["text_norm"] = (df.get("subject","").fillna("").astype(str) + " " +
                   df.get("body","").fillna("").astype(str) + " " +
                   df.get("pr_title","").fillna("").astype(str)).apply(norm_text)
df["tokens"]  = df["text_norm"].apply(toks)
df["bigrams"] = df["tokens"].apply(bigr)

token_index  = defaultdict(list)
bigram_index = defaultdict(list)
for i, row in df.iterrows():
    for t in set(row["tokens"]): token_index[t].append(i)
    for bg in set(row["bigrams"]): bigram_index[bg].append(i)

MIN_UNIGRAM_FREQ = 8
MIN_BIGRAM_FREQ  = 4
STOP_WORDS = set([
    "a","an","the","of","for","on","in","to","and","or","if","by","with","from","as","at","into","about","over","under","up","down",
    "that","this","those","these","is","are","was","were","be","been","being","it","its","your","our","their","we","you","they","i","me","my",
    "mine","ours","yours","his","her","hers","theirs","not","no","yes","true","false","null","none","na","todo","wip","pr","ci","build","test","tests","run"
])

def clean_term(t: str):
    t = str(t).strip().lower()
    t = re.sub(r"\s+", " ", t)
    return t

cue_terms = [clean_term(x) for x in cues.get("cue", pd.Series([])).dropna().astype(str).tolist()]

uni_terms = []
if set(["token","count"]).issubset(uni.columns):
    uni_filt = uni[uni["count"] >= MIN_UNIGRAM_FREQ].copy()
    for tok in uni_filt["token"].astype(str):
        tt = clean_term(tok)
        if len(tt) < 2 or tt in STOP_WORDS: continue
        uni_terms.append(tt)

bi_terms = []
if set(["bigram","count"]).issubset(bi.columns):
    bi_filt = bi[bi["count"] >= MIN_BIGRAM_FREQ].copy()
    for bg in bi_filt["bigram"].astype(str):
        bb = clean_term(bg)
        if len(bb.split()) != 2: continue
        bi_terms.append(bb)

vocab_terms = list(dict.fromkeys(cue_terms + uni_terms + bi_terms))

term_freq = {}
if "rows_with_cue" in cues.columns:
    for cue, cnt in cues[["cue","rows_with_cue"]].itertuples(index=False):
        term_freq[clean_term(cue)] = int(cnt)
if set(["token","count"]).issubset(uni.columns):
    for tok, cnt in uni[["token","count"]].itertuples(index=False):
        term_freq.setdefault(clean_term(tok), int(cnt))
if set(["bigram","count"]).issubset(bi.columns):
    for bg, cnt in bi[["bigram","count"]].itertuples(index=False):
        term_freq.setdefault(clean_term(bg), int(cnt))

def rows_for_term(term: str):
    tt = toks(term)
    if len(tt) == 1: return token_index.get(tt[0], [])
    if len(tt) == 2: return bigram_index.get(f"{tt[0]} {tt[1]}", [])
    return []

MAX_TERMS = 2000
THRESH = 0.25
terms_sorted = sorted(vocab_terms, key=lambda t: term_freq.get(t, 0), reverse=True)[:MAX_TERMS]

incidence = {}
for term in terms_sorted:
    idxs = rows_for_term(term)
    if idxs: incidence[term] = set(idxs)

leaders, assign = [], {}
def jaccard(a: set, b: set):
    if not a or not b: return 0.0
    inter = len(a & b)
    return 0.0 if inter == 0 else inter / len(a | b)

for term in sorted(incidence.keys(), key=lambda t: len(incidence[t]), reverse=True):
    placed = False
    for cid, lead_term in enumerate(leaders):
        if jaccard(incidence[term], incidence[lead_term]) >= THRESH:
            assign[term] = cid; placed = True; break
    if not placed:
        leaders.append(term); assign[term] = len(leaders)-1

rows = []
for term, cid in assign.items():
    rows.append({
        "term": term,
        "source_type": ("bigram" if " " in term else ("cue" if term in cue_terms else "unigram")),
        "frequency": term_freq.get(term, 0),
        "cluster_id": cid
    })
term_df = pd.DataFrame(rows).sort_values(["cluster_id","frequency"], ascending=[True, False])
term_df.to_csv(P_IND_TERMS, index=False)

cluster_summary = []
def example_subjects_for_term(term: str, k=5):
    tt = toks(term)
    if len(tt) == 1:
        idxs = token_index.get(tt[0], [])[:k]
    elif len(tt) == 2:
        idxs = bigram_index.get(f"{tt[0]} {tt[1]}", [])[:k]
    else:
        idxs = []
    subs = (df.loc[idxs, "subject"].fillna("").astype(str)).tolist()
    return " | ".join([s[:160] for s in subs])

for cid, grp in term_df.groupby("cluster_id"):
    top_terms = grp.head(12)["term"].tolist()
    ex = []
    for t in top_terms[:3]:
        ex.append(example_subjects_for_term(t, k=3))
    cluster_summary.append({
        "cluster_id": cid,
        "size": len(grp),
        "top_terms_by_freq": ", ".join(top_terms),
        "example_subjects": " | ".join([e for e in ex if e][:6])
    })
summary_df = pd.DataFrame(cluster_summary).sort_values("size", ascending=False)
summary_df.to_csv(P_IND_SUM, index=False)

P_IND_TERMS, P_IND_SUM, term_df.shape, summary_df.shape

(WindowsPath('C:/Android Mobile App/Step2_Clone_Repo/Type_1/Aug_10/RQ2/Second_Label/Method_V1.0/2_secondary_label_inductive_clusters.csv'),
 WindowsPath('C:/Android Mobile App/Step2_Clone_Repo/Type_1/Aug_10/RQ2/Second_Label/Method_V1.0/2_secondary_label_inductive_cluster_summary.csv'),
 (1999, 4),
 (1036, 4))

## 3. Canonical naming — codebook template (CSV & YAML)

In [63]:
import pandas as pd

term_df = pd.read_csv(P_IND_TERMS)
summary_df = pd.read_csv(P_IND_SUM)

codebook_rows = []
for r in summary_df.itertuples(index=False):
    cid = int(r.cluster_id)
    codebook_rows.append({
        "cluster_id": cid,
        "proposed_canonical_name": "",
        "description": "",
        "regex_positive_cues": "",
        "regex_context_guards": "",
        "top_terms": getattr(r, "top_terms_by_freq", ""),
        "example_subjects": getattr(r, "example_subjects", ""),
        "notes": ""
    })
codebook = pd.DataFrame(codebook_rows).sort_values("cluster_id")
codebook.to_csv(P_CODEBOOK_CSV, index=False)

def to_yaml(rows):
    lines = ["clusters:"]
    for r in rows:
        lines.append(f"  - cluster_id: {r['cluster_id']}")
        lines.append(f"    proposed_canonical_name: \"{r['proposed_canonical_name']}\"")
        lines.append(f"    description: \"{r['description']}\"")
        lines.append(f"    regex_positive_cues: \"{r['regex_positive_cues']}\"")
        lines.append(f"    regex_context_guards: \"{r['regex_context_guards']}\"")
        lines.append(f"    top_terms: \"{r['top_terms']}\"")
        lines.append(f"    example_subjects: \"{r['example_subjects']}\"")
        lines.append(f"    notes: \"{r['notes']}\"")
    return "\n".join(lines) + "\n"

with open(P_CODEBOOK_YAML, "w", encoding="utf-8") as f:
    f.write(to_yaml(codebook_rows))

term_df[["cluster_id","term","source_type","frequency"]].to_csv(P_TERMS_CSV, index=False)

P_CODEBOOK_CSV, P_CODEBOOK_YAML, P_TERMS_CSV

(WindowsPath('C:/Android Mobile App/Step2_Clone_Repo/Type_1/Aug_10/RQ2/Second_Label/Method_V1.0/3_secondary_label_codebook_template.csv'),
 WindowsPath('C:/Android Mobile App/Step2_Clone_Repo/Type_1/Aug_10/RQ2/Second_Label/Method_V1.0/3_secondary_label_codebook_template.yaml'),
 WindowsPath('C:/Android Mobile App/Step2_Clone_Repo/Type_1/Aug_10/RQ2/Second_Label/Method_V1.0/3_secondary_label_clusters_terms.csv'))

Auto Regex after the Manual Review

In [64]:
import re
import pandas as pd
from pathlib import Path
from collections import Counter

# ========= CONFIG =========
IN_PATH  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V1.0\3_secondary_label_codebook_filled_v2.csv")
OUT_PATH = IN_PATH.with_name("3_secondary_label_codebook_autoregex.csv")
VAL_PATH = IN_PATH.with_name("codebook_autoregex_validation.csv")

# knobs (tune if you like)
MAX_SINGLE = 10   # top-N single tokens to include in the regex
MAX_BIGRAM = 8    # top-N bigrams (from example subjects) to include
MIN_TOKEN_LEN = 3

# ========= HELPERS =========
STOP = set("""
a an the of for on in to and or if by with from as at into about over under up down
that this those these is are was were be been being it its your our their we you they i me my
mine ours yours his her hers theirs not no yes true false null none na todo wip pr ci build test tests run
""".split())

def norm(s: str) -> str:
    s = str(s or "").lower()
    s = re.sub(r"[^\w\s\./-]+", " ", s)  # keep word chars, spaces, dot, slash, hyphen
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokens(s: str):
    return re.findall(r"[a-z0-9_]+", norm(s))

def bigrams(tok_list):
    return ["{} {}".format(tok_list[i], tok_list[i+1]) for i in range(len(tok_list)-1)]

def split_terms_csv(s: str):
    # "top_terms" often looks like "term1, term2, term3"
    return [p.strip() for p in (s or "").split(",") if p.strip()]

def is_signal_token(t: str) -> bool:
    if not t: return False
    if len(t) < MIN_TOKEN_LEN: return False
    if t in STOP: return False
    return True

def rx_escape_token(t: str) -> str:
    # keep underscores as word chars; add word boundaries
    return r"\b" + re.escape(t).replace(r"\_", "_") + r"\b"

def rx_escape_bigram(bg: str) -> str:
    a, b = bg.split(" ", 1)
    return r"\b" + re.escape(a).replace(r"\_", "_") + r"\s+" + re.escape(b).replace(r"\_", "_") + r"\b"

def try_compile(rx: str):
    rx = str(rx or "").strip()
    if not rx: return True, ""
    try:
        re.compile(rx, re.I)
        return True, ""
    except re.error as e:
        return False, str(e)

def build_regex_from_row(row, max_single=MAX_SINGLE, max_bigram=MAX_BIGRAM):
    """
    Build two regex strings *purely* from this row’s content:
      - regex_positive_cues: OR of the top bigrams (from example subjects) + top single tokens
      - regex_context_guards: optional docs guard if docs tokens dominate examples
    Also returns the lists of tokens/bigrams used (for transparency).
    """
    name = str(row.get("proposed_canonical_name", "") or "")
    desc = str(row.get("description", "") or "")
    top  = str(row.get("top_terms", "") or "")
    ex   = str(row.get("example_subjects", "") or "")

    # candidate single tokens (name → split on "_" and space; description; top_terms; examples)
    cand_tokens = []
    cand_tokens += [t for t in re.split(r"[\s_]+", norm(name)) if t]     # name
    cand_tokens += tokens(desc)                                         # description
    for part in split_terms_csv(top):                                   # top_terms (CSV-like)
        cand_tokens += tokens(part)
    cand_tokens += tokens(ex)                                           # example subjects

    cand_tokens = [t for t in cand_tokens if is_signal_token(t)]
    tok_counts = Counter(cand_tokens)

    # candidate bigrams from example subjects (tend to be precise)
    ex_toks = tokens(ex)
    ex_bi = [bg for bg in bigrams(ex_toks) if all(is_signal_token(z) for z in bg.split())]
    bi_counts = Counter(ex_bi)

    # choose top-N
    top_single = [t for t, _ in tok_counts.most_common(max_single)]
    top_bigram = [bg for bg, _ in bi_counts.most_common(max_bigram)]

    # build OR regex: bigrams first, then singles (each wrapped as non-capturing group)
    rx_parts = [rx_escape_bigram(bg) for bg in top_bigram] + [rx_escape_token(t) for t in top_single]
    if not rx_parts:
        return "", "", [], []

    rx_pos = "|".join(f"(?:{p})" for p in rx_parts)

    # optional docs guard (only if docs terms are frequent in examples)
    ex_doc_tokens = {"doc", "docs", "documentation", "readme", "changelog"}
    doc_hits = sum(1 for t in ex_toks if t in ex_doc_tokens)
    rx_guard = ""
    if doc_hits >= max(2, int(0.4 * (len(ex_toks) or 0))):
        rx_guard = r"\b(doc|docs|documentation|readme|changelog)\b"

    return rx_pos, rx_guard, top_single, top_bigram

# ========= MAIN =========
df = pd.read_csv(IN_PATH)
df.columns = [c.strip().lower() for c in df.columns]

# be tolerant if any optional columns are missing
for need in ["proposed_canonical_name","description","top_terms","example_subjects"]:
    if need not in df.columns:
        df[need] = ""

out = df.copy()
pos_list, guard_list, used_single_list, used_bigram_list = [], [], [], []

for _, r in df.iterrows():
    rx_pos, rx_guard, singles_used, bigrams_used = build_regex_from_row(r)
    pos_list.append(rx_pos)
    guard_list.append(rx_guard)
    used_single_list.append(";".join(singles_used))
    used_bigram_list.append(";".join(bigrams_used))

out["regex_positive_cues"] = pos_list
out["regex_context_guards"] = guard_list
# transparency/debug columns — optional, remove if you don’t want them
out["regex_used_singles"] = used_single_list
out["regex_used_bigrams"] = used_bigram_list

# validate the generated regexes compile
val_rows = []
for i, r in out.iterrows():
    ok_pos, err_pos = try_compile(r["regex_positive_cues"])
    ok_guard, err_guard = try_compile(r["regex_context_guards"])
    val_rows.append({
        "row": i,
        "cluster_id": r.get("cluster_id", ""),
        "proposed_canonical_name": r.get("proposed_canonical_name", ""),
        "ok_positive": ok_pos, "error_positive": err_pos,
        "ok_guard": ok_guard, "error_guard": err_guard
    })
val = pd.DataFrame(val_rows)

out.to_csv(OUT_PATH, index=False, encoding="utf-8")
val.to_csv(VAL_PATH, index=False, encoding="utf-8")

print("Wrote:", OUT_PATH)
print("Validation:", VAL_PATH)
print("Rows:", len(out),
      "| non-empty positive regex:", int((out["regex_positive_cues"].astype(str).str.strip()!="").sum()),
      "| non-empty guards:", int((out["regex_context_guards"].astype(str).str.strip()!="").sum()))


Wrote: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V1.0\3_secondary_label_codebook_autoregex.csv
Validation: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V1.0\codebook_autoregex_validation.csv
Rows: 1036 | non-empty positive regex: 1036 | non-empty guards: 0


## 4. Search rules — compile your codebook regex & label commits

In [65]:
import re, pandas as pd

codebook = pd.read_csv(P_CODEBOOK_CSV_F)
commits  = pd.read_csv(P_SAMPLE)

compiled = []
for r in codebook.itertuples(index=False):
    cid = int(getattr(r, "cluster_id"))
    name = str(getattr(r, "proposed_canonical_name") or "").strip()
    pos  = str(getattr(r, "regex_positive_cues") or "").strip()
    guards = str(getattr(r, "regex_context_guards") or "").strip()
    if not name or not pos: 
        continue
    rx_pos = re.compile(pos, re.I)
    rx_guard = re.compile(guards, re.I) if guards else None
    compiled.append((name, rx_pos, rx_guard))

def normalize_text(s: str) -> str:
    s = str(s or "")
    s = s.lower()
    s = re.sub(r"[^\w\s\./-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

commits["text_norm"] = (commits.get("subject","").fillna("").astype(str) + " " +
                        commits.get("body","").fillna("").astype(str) + " " +
                        commits.get("pr_title","").fillna("").astype(str)).apply(normalize_text)

def detect_labels(text: str):
    labels = []
    for name, rx_pos, rx_guard in compiled:
        if rx_pos.search(text or ""):
            if rx_guard and rx_guard.search(text or ""):
                continue
            labels.append(name)
    return ";".join(sorted(set(labels)))

commits["pred_secondary_labels"] = commits["text_norm"].apply(detect_labels)
commits.to_csv(P_DETECTIONS, index=False)
print(f"Wrote detections to {P_DETECTIONS}")
commits[["repo","sha","subject","pred_secondary_labels"]].head(12)

Wrote detections to C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V1.0\4_commit_secondary_label_predictions.csv


,repo,sha,subject,pred_secondary_labels
0,dlew/joda-time-android,659c4d59a5688f094dfd4cf9ccc996cdf8827a03,Added gradle-based builds,cleanup;expand_coverage;fix_build;migrate_ci;n...
1,dlew/joda-time-android,ec1ca1c8ba36a77befd85828100f531355e51760,Added gradle wrapper,cleanup;expand_coverage;fix_build;migrate_ci;n...
2,dlew/joda-time-android,2a4cd741661eae02054c54b132c8324eb084ad0f,Gradle check now works,cleanup;expand_coverage;fix_build;migrate_ci;u...
3,dlew/joda-time-android,d96b46527e3adc63f5932535dd8fe5911a8c3ba2,"Added gradle-mvn-push support, for pushing aar",cleanup;expand_coverage;fix_build;migrate_ci;n...
4,dlew/joda-time-android,d96b46527e3adc63f5932535dd8fe5911a8c3ba2,"Added gradle-mvn-push support, for pushing aar",cleanup;expand_coverage;fix_build;migrate_ci;n...
5,dlew/joda-time-android,4d5aa308f31be6d62e4f95ea51419eedd5ef45e0,Add task and instructions for including update...,cleanup;expand_coverage;fix_build;migrate_ci;n...
6,dlew/joda-time-android,5e3953840a767a610aec1965aa8ecdbfcff6e313,Updated gradle/plugin versions,cleanup;expand_coverage;fix_build;migrate_ci;n...
7,dlew/joda-time-android,8f365ca1d6e2bb22c3ff909ec57a1a6cabb17652,Updated gradle wrapper files,cleanup;expand_coverage;fix_build;migrate_ci;n...
8,dlew/joda-time-android,0eccbe0132e134768b56864c497258369d40f619,Prepare release v2.10.14,cleanup;expand_coverage;fix_build;flake_mitiga...
9,dlew/joda-time-android,5f8ec9af356e2e9d1b5ec3a913945bf5ac3f4b83,Pull joda-convert from Maven Central,expand_coverage;migrate_ci;nan;upgrade


## 5. Refinement — metrics if ground truth exists; otherwise error surfacing

In [66]:
import pandas as pd

df = pd.read_csv(P_DETECTIONS)
df.columns = [c.strip().lower() for c in df.columns]

truth_col = None
for cand in ["secondary_labels","true_secondary_labels","labels","pred_secondary_labels"]:
    if cand in df.columns:
        truth_col = cand; break

if truth_col:
    def to_set(s):
        if pd.isna(s) or not str(s).strip(): return set()
        return set([t.strip() for t in str(s).split(";") if t.strip()])
    df["truth"] = df[truth_col].apply(to_set)
    df["pred"]  = df["pred_secondary_labels"].apply(to_set)
    labels = sorted(set().union(*df["truth"].tolist(), *df["pred"].tolist()))
    rows = []
    for lab in labels:
        tp = sum(1 for r in df.itertuples(index=False) if (lab in r.truth and lab in r.pred))
        fp = sum(1 for r in df.itertuples(index=False) if (lab not in r.truth and lab in r.pred))
        fn = sum(1 for r in df.itertuples(index=False) if (lab in r.truth and lab not in r.pred))
        prec = tp/(tp+fp) if tp+fp else 0.0
        rec  = tp/(tp+fn) if tp+fn else 0.0
        f1   = 0.0 if (prec+rec)==0 else 2*prec*rec/(prec+rec)
        rows.append({"label": lab, "TP": tp, "FP": fp, "FN": fn, 
                     "precision": round(prec,3), "recall": round(rec,3), "f1": round(f1,3)})
    metrics = pd.DataFrame(rows).sort_values(["f1","label"], ascending=[False, True])
    metrics.to_csv(P_EVAL, index=False)
    print(f"Metrics written to {P_EVAL}")
    metrics.head(20)
else:
    df["n_pred"] = df["pred_secondary_labels"].apply(lambda s: 0 if pd.isna(s) or not str(s).strip() else len(str(s).split(";")))
    samples_no = df[df["n_pred"]==0].head(50)[["repo","sha","subject","pred_secondary_labels"]]
    samples_multi = df[df["n_pred"]>=2].head(50)[["repo","sha","subject","pred_secondary_labels"]]
    samples_no.to_csv(BASE / "inspection_no_label_examples.csv", index=False)
    samples_multi.to_csv(BASE / "inspection_multi_label_examples.csv", index=False)
    print("No ground truth column found. Wrote inspection samples.")

Metrics written to C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V1.0\5_secondary_label_evaluation.csv
